# DE0831 Research Project, topic 10: live demo on Google Colab — Vagif Asgarli, 251ADM047

Before the talk (about 15 minutes): Runtime → Change runtime type → **Python 3, T4 GPU**; run steps 1–4 (install Julia, upload `DE0831_RP.zip`, instantiate) and the **warm-up** cell of step 5, which generates the 125,000-line corpus and compiles the kernels. During the talk run only the **live run** cell (about one minute): validation on 125,000 lines (`Correct!`), then the GPU and CPU matchers at 10, 1,000 and 10,000 indicators.


## 1. GPU

In [ ]:
!nvidia-smi

## 2. Install Julia 1.12.7 (the version the Manifest was resolved with)

In [ ]:
%%bash
set -e
cd /content
if [ ! -x /content/julia-1.12.7/bin/julia ]; then
  wget -q https://julialang-s3.julialang.org/bin/linux/x64/1.12/julia-1.12.7-linux-x86_64.tar.gz
  tar xzf julia-1.12.7-linux-x86_64.tar.gz
fi
/content/julia-1.12.7/bin/julia --version

## 3. Upload the project (`DE0831_RP.zip`)

In [ ]:
import os, zipfile
from google.colab import files
print('Choose DE0831_RP.zip. A newer zip can be uploaded over an existing /content/DE0831_RP:',
      'its source files are replaced and the generated data/ is kept.')
uploaded = files.upload()
for name in uploaded:
    zipfile.ZipFile(name).extractall('/content')
    print('extracted', name)
assert os.path.isdir('/content/DE0831_RP'), 'DE0831_RP.zip was not uploaded'
print(sorted(os.listdir('/content/DE0831_RP')))


## 4. Install packages (CUDA.jl, CairoMakie) and check CUDA

In [ ]:
import os, subprocess

def run(cmd, log):
    """Run a shell command, show its output live in the cell and save it to a log file."""
    os.makedirs('/content/logs', exist_ok=True)
    with open(log, 'w') as f:
        p = subprocess.Popen(cmd, shell=True, executable='/bin/bash', text=True,
                             stdout=subprocess.PIPE, stderr=subprocess.STDOUT)
        for line in p.stdout:
            print(line, end='', flush=True)
            f.write(line)
        p.wait()
    assert p.returncode == 0, f'command failed (exit {p.returncode}), see the output above or {log}'


In [ ]:
run("cd /content/DE0831_RP && /content/julia-1.12.7/bin/julia --project=. -e 'using Pkg; Pkg.instantiate(); using CUDA; CUDA.versioninfo(); @assert CUDA.functional()'", '/content/logs/instantiate.log')

## 5. Live demo


In [ ]:
# warm-up before the talk: generate the 125k corpus, compile the kernels (RP_QUICK=1 = validation only)
run('cd /content/DE0831_RP && RP_QUICK=1 RP_MAIN=125000 RP_LISTS=10,1000,10000 /content/julia-1.12.7/bin/julia -t auto --project=. src/rp_run.jl', '/content/logs/warmup.log')

In [ ]:
# live run: validation, then the list-size sweep at 10, 1,000 and 10,000 indicators on 125k lines (about one minute)
run('cd /content/DE0831_RP && RP_MAIN=125000 RP_LISTS=10,1000,10000 RP_SIZES=125000 RP_SIZE_LIST=1000 RP_REPS=20 RP_CPU_REPS=1 RP_G1_MAX=1000 RP_C2_MAX=10 RP_NO_PLOTS=1 /content/julia-1.12.7/bin/julia -t auto --project=. src/rp_run.jl', '/content/logs/demo.log')

In [ ]:
from IPython.display import Markdown, display
display(Markdown(open('/content/DE0831_RP/results/benchmarks.md').read().split('### Corpus-size sweep')[0]))